<a href="https://colab.research.google.com/github/amorimriki/Projeto-I-PSA/blob/main/JUPYTER_NOTEBOOK/model_for_GoogleColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# --------------------------------------------------------------
# Dependencies
# --------------------------------------------------------------
!pip install statsmodels

import pandas as pd
import pylab as pl
import numpy as np
import scipy.optimize as opt
import statsmodels.api as sm

from google.colab import files


import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
import seaborn as sns


import itertools

import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.9/232.9 kB 14.4 MB/s eta 0:00:00


In [5]:
# --------------------------------------------------------------
# Import Dataset
# --------------------------------------------------------------


df = pd.read_csv("/content/student_data.csv")



In [ ]:


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from google.colab import files

# Features e target
target = 'final_result'
X = df.drop(columns=[target])
y = df[target]

# Features categóricas e numéricas
categorical_features = ['code_module', 'gender', 'region', 'highest_education',
                        'imd_band', 'age_band', 'disability', 'assessment_type', 'is_banked']
numerical_features = ['date_submitted', 'num_of_prev_attempts', 'sum_click',
                      'date', 'studied_credits', 'weight', 'score']

# Preprocessamento
categorical_transformer = OneHotEncoder(handle_unknown='ignore')
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numerical_transformer, numerical_features)
    ]
)

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =====================
# Random Forest Pipeline + GridSearchCV
# =====================
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
print("Melhor Random Forest:")
print(grid_rf.best_params_)
print(classification_report(y_test, best_rf.predict(X_test)))

# =====================
# SVM Pipeline + GridSearchCV
# =====================
svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True))
])

param_grid_svm = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(svm_pipeline, param_grid_svm, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_svm.fit(X_train, y_train)
best_svm = grid_svm.best_estimator_
print("Melhor SVM:")
print(grid_svm.best_params_)
print(classification_report(y_test, best_svm.predict(X_test)))

# =====================
# MLP Pipeline + GridSearchCV
# =====================
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', MLPClassifier(max_iter=300, random_state=42))
])

param_grid_mlp = {
    'classifier__hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__solver': ['adam'],
    'classifier__alpha': [0.0001, 0.001],
    'classifier__learning_rate': ['constant', 'adaptive']
}

grid_mlp = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_mlp.fit(X_train, y_train)
best_mlp = grid_mlp.best_estimator_
print("Melhor MLP:")
print(grid_mlp.best_params_)
print(classification_report(y_test, best_mlp.predict(X_test)))

# =====================
# Ensemble com VotingClassifier
# =====================
ensemble = VotingClassifier(
    estimators=[
        ('rf', best_rf),
        ('svm', best_svm),
        ('mlp', best_mlp)
    ],
    voting='soft'
)

ensemble.fit(X_train, y_train)
y_pred_ensemble = ensemble.predict(X_test)

print("Avaliação do Ensemble:")
print(classification_report(y_test, y_pred_ensemble))
print(confusion_matrix(y_test, y_pred_ensemble))

# Guardar modelo ensemble
joblib.dump(ensemble, 'ensemble_model.pkl')
files.download('ensemble_model.pkl')


Fitting 3 folds for each of 24 candidates, totalling 72 fits
